In [4]:
import pyreadstat

path = r"C:\Users\ouame\Downloads\2021-2022\UKDA-9136-spss\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav"

# metadataonly=True reads just the column info, not the full 1.8GB - fast and safe
df, meta = pyreadstat.read_sav(path, metadataonly=True)

print(f"Total columns: {len(meta.column_names)}")
print(f"Total rows: {meta.number_rows}\n")

for name, label in meta.column_names_to_labels.items():
    print(f"{name:20s} | {label}")

Total columns: 10488
Total rows: 177551

serial               | Serial
wt_final             | All: Online and Postal
wt_final_AB          | Postal & Online Group 1
wt_final_AC          | Postal & Online Group 2
wt_final_B           | Online Group 1
wt_final_C           | Online Group 2
wt_final_online      | All: Online
wt_online_time       | All: Online - time series weight
wt_time              | All: Online and Postal - time series weight
xStrata              | Copy of variable LA for use as strata
Overall              | All adults (aged 16 and over)
Age16plus            | Aged 16 and over
Age19plus            | Aged 19 and over
group                | Question Rotation
mode                 | Mode of completion
Quarter              | Survey quarter (from mid-month to mid-month)
month                | Interview month
MonthTwelve          | Interview month, coded 1-12
month_gend           | Interview month by gender
month_gendage1660    | Month by gender by age 16-60
Month_GR6          

In [5]:
import pyreadstat

path = r"C:\Users\ouame\Downloads\2021-2022\UKDA-9136-spss\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav"

# load just the columns we need, with labels applied
cols = ["serial", "wt_final", "Reg9", "LA_2021", "ACT7GR_ALL", "Age9"]
df, meta = pyreadstat.read_sav(path, usecols=cols, apply_value_formats=True)

print(df["Reg9"].value_counts())
print(df["ACT7GR_ALL"].value_counts())
print(df["LA_2021"].value_counts().head(40))

Reg9
South East                  32283
North West                  27111
East                        22533
East Midlands               19268
West Midlands               19066
Yorkshire and the Humber    16982
London                      16382
South West                  16366
North East                   7560
Name: count, dtype: int64
ACT7GR_ALL
Inactive - nothing       21305
Inactive - light only    15408
Inactive - 1-29 mins      1946
Name: count, dtype: int64
LA_2021
E08000017 Doncaster                      3529
E08000025 Birmingham                     2936
E08000003 Manchester                     2063
E08000019 Sheffield                      2035
E08000021 Newcastle upon Tyne            2019
E06000018 Nottingham                     2012
E08000012 Liverpool                      2011
E08000035 Leeds                          2003
E06000023 Bristol, City of               1985
E08000008 Tameside                       1039
E08000005 Rochdale                       1024
E08000026 Coventry 

In [6]:
import pyreadstat

path = r"C:\Users\ouame\Downloads\2021-2022\UKDA-9136-spss\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav"

cols = ["MEMS7GR_ALL", "ACT7GR_ALL"]
df, meta = pyreadstat.read_sav(path, usecols=cols, apply_value_formats=True)

print("MEMS7GR_ALL:")
print(df["MEMS7GR_ALL"].value_counts())
print("\nACT7GR_ALL (non-null count):", df["ACT7GR_ALL"].notna().sum(), "out of", len(df))

MEMS7GR_ALL:
MEMS7GR_ALL
Active           120501
Inactive          38659
Fairly Active     18391
Name: count, dtype: int64

ACT7GR_ALL (non-null count): 38659 out of 177551


In [8]:
import pyreadstat
import pandas as pd

PATH = r"C:\Users\ouame\Downloads\2021-2022\UKDA-9136-spss\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav"
SURVEY_YEAR_LABEL = "2021-22"

cols = ["Reg9", "LA_2021", "wt_final", "MEMS7GR_ALL"]
df, meta = pyreadstat.read_sav(PATH, usecols=cols, apply_value_formats=True)

# --- filter to London only ---
london = df[df["Reg9"] == "London"].copy()

# --- clean borough name: strip the "E09000XX " ONS code prefix ---
london["borough"] = london["LA_2021"].str.replace(r"^E\d{8}\s+", "", regex=True)

# --- weighted percentage of each activity category, per borough ---
def weighted_pct(group, category):
    mask = group["MEMS7GR_ALL"] == category
    return 100 * group.loc[mask, "wt_final"].sum() / group["wt_final"].sum()

rows = []
for borough, group in london.groupby("borough"):
    rows.append({
        "survey_year": SURVEY_YEAR_LABEL,
        "borough": borough,
        "pct_active": round(weighted_pct(group, "Active"), 1),
        "pct_fairly_active": round(weighted_pct(group, "Fairly Active"), 1),
        "pct_inactive": round(weighted_pct(group, "Inactive"), 1),
        "respondents": len(group),
        "weighted_base": round(group["wt_final"].sum(), 1),
    })

gap_score = pd.DataFrame(rows).sort_values("borough").reset_index(drop=True)

gap_score.to_csv("active_lives_2021-22_london_gapscore.csv", index=False)
print(f"Boroughs found: {len(gap_score)} (should be 33)")
gap_score

Boroughs found: 33 (should be 33)


,survey_year,borough,pct_active,pct_fairly_active,pct_inactive,respondents,weighted_base
0,2021-22,Barking and Dagenham,56.6,7.5,35.9,519,604.3
1,2021-22,Barnet,65.5,12.4,22.1,506,1244.6
2,2021-22,Bexley,63.6,11.9,24.5,487,764.1
3,2021-22,Brent,59.9,11.3,28.8,512,1010.1
4,2021-22,Bromley,73.5,7.6,18.9,491,1038.9
5,2021-22,Camden,71.2,9.5,19.3,503,906.6
6,2021-22,City of London,36.8,20.1,43.1,243,35.9
7,2021-22,Croydon,63.0,13.0,24.0,497,1175.4
8,2021-22,Ealing,66.8,11.5,21.7,485,1040.3
9,2021-22,Enfield,61.6,9.2,29.2,506,998.9


In [9]:
MIN_SAMPLE_SIZE = 30  # flag (not drop) cells below this - team to confirm threshold

DEMOGRAPHIC_VARS = [
    "Age9", "Gend3", "Eth7", "IMD10", "Disab3", "LondInOut",
    "NSSEC5", "Educ6", "Orient4", "Relig7", "ChildAgeU13", "Maternity_pop",
]

cols2 = ["Reg9", "LA_2021", "wt_final", "MEMS7GR_ALL"] + DEMOGRAPHIC_VARS
df2, meta2 = pyreadstat.read_sav(PATH, usecols=cols2, apply_value_formats=True)

london2 = df2[df2["Reg9"] == "London"].copy()
london2["borough"] = london2["LA_2021"].str.replace(r"^E\d{8}\s+", "", regex=True)

def weighted_pct2(group, category):
    total_wt = group["wt_final"].sum()
    if total_wt == 0:
        return None
    mask = group["MEMS7GR_ALL"] == category
    return round(100 * group.loc[mask, "wt_final"].sum() / total_wt, 1)

def build_row(borough_label, dem_var, dem_category, group):
    return {
        "survey_year": SURVEY_YEAR_LABEL,
        "borough": borough_label,
        "demographic_group": dem_var,
        "category": dem_category,
        "pct_active": weighted_pct2(group, "Active"),
        "pct_fairly_active": weighted_pct2(group, "Fairly Active"),
        "pct_inactive": weighted_pct2(group, "Inactive"),
        "respondents": len(group),
        "weighted_base": round(group["wt_final"].sum(), 1),
        "suppress": len(group) < MIN_SAMPLE_SIZE,
    }

rows2 = []
for dem_var in DEMOGRAPHIC_VARS:
    valid = london2[london2[dem_var].notna()]

    for dem_category, group in valid.groupby(dem_var):
        rows2.append(build_row("London", dem_var, dem_category, group))

    for (borough, dem_category), group in valid.groupby(["borough", dem_var]):
        rows2.append(build_row(borough, dem_var, dem_category, group))

population_profile = pd.DataFrame(rows2)
population_profile.to_csv("activelives_population_profile_2021-22.csv", index=False)

print("Total rows:", len(population_profile))
print("Suppressed rows:", population_profile["suppress"].sum())
population_profile.head(20)

C:\Users\ouame\AppData\Local\Temp\ipykernel_16584\2274209033.py:39: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for dem_category, group in valid.groupby(dem_var):
C:\Users\ouame\AppData\Local\Temp\ipykernel_16584\2274209033.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (borough, dem_category), group in valid.groupby(["borough", dem_var]):
C:\Users\ouame\AppData\Local\Temp\ipykernel_16584\2274209033.py:39: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the

Total rows: 1885
Suppressed rows: 665


,survey_year,borough,demographic_group,category,pct_active,pct_fairly_active,pct_inactive,respondents,weighted_base,suppress
0,2021-22,London,Age9,16-24,67.6,9.7,22.7,1309,3670.6,False
1,2021-22,London,Age9,25-34,71.2,10.2,18.6,3091,6171.9,False
2,2021-22,London,Age9,35-44,66.6,10.6,22.8,3428,5630.9,False
3,2021-22,London,Age9,45-54,68.1,10.5,21.4,2753,4389.6,False
4,2021-22,London,Age9,55-64,66.8,9.2,24.0,2482,3475.3,False
5,2021-22,London,Age9,65-74,66.4,10.2,23.4,1961,2346.3,False
6,2021-22,London,Age9,75-84,51.0,12.8,36.3,977,1512.4,False
7,2021-22,London,Age9,85+,32.5,13.5,54.0,245,385.0,False
8,2021-22,Barking and Dagenham,Age9,16-24,52.9,12.5,34.6,58,105.3,False
9,2021-22,Barking and Dagenham,Age9,25-34,61.2,5.4,33.4,98,125.8,False
